## Blend Or Stack
**After watching top kernal many of them are blend and stacking.**

This show how the scores can be imporved by add more models.
Thanks for show this kernals.

ref:
1. https://www.kaggle.com/danmusetoiu/staking-and-stealing-like-a-molecule
2. https://www.kaggle.com/marcelotamashiro/lgb-public-kernels-plus-more-features
3. https://www.kaggle.com/rajwardhanshinde/blend-of-blends
4. https://www.kaggle.com/vaishvik25/blend



## <font color=blue>Vote early and vote often!</font>


## Stat Stack

In [1]:
import os
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from subprocess import check_output
print(check_output(["ls", "../input"]).decode("utf8"))

champs-scalar-coupling
description.md
dipole_moments.csv
dipole_moments.csv.zip
magnetic_shielding_tensors.csv
magnetic_shielding_tensors.csv.zip
mulliken_charges.csv
mulliken_charges.csv.zip
potential_energy.csv
potential_energy.csv.zip
sample_submission.csv
sample_submission.csv.zip
scalar_coupling_contributions.csv
scalar_coupling_contributions.csv.zip
structures
structures.csv
structures.csv.zip
structures.zip
test.csv
test.csv.zip
train.csv
train.csv.zip



In [2]:
sub_path = "../input/top-mol"
all_files = os.listdir(sub_path)
all_files

FileNotFoundError: [Errno 2] No such file or directory: '../input/top-mol'

In [3]:
outs = [pd.read_csv(os.path.join(sub_path, f), index_col=0) for f in all_files]
concat_sub = pd.concat(outs, axis=1)
cols = list(map(lambda x: "mol" + str(x), range(len(concat_sub.columns))))
concat_sub.columns = cols
concat_sub.reset_index(inplace=True)
concat_sub.head()
ncol = concat_sub.shape[1]

NameError: name 'all_files' is not defined

In [4]:
# check correlation
concat_sub.iloc[:,1:ncol].corr()

NameError: name 'concat_sub' is not defined

In [5]:
corr = concat_sub.iloc[:,1:7].corr()
mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})

NameError: name 'concat_sub' is not defined

In [6]:
# get the data fields ready for stacking
concat_sub['m_max'] = concat_sub.iloc[:, 1:ncol].max(axis=1)
concat_sub['m_min'] = concat_sub.iloc[:, 1:ncol].min(axis=1)
concat_sub['m_mean'] = concat_sub.iloc[:, 1:ncol].mean(axis=1)
concat_sub['m_median'] = concat_sub.iloc[:, 1:ncol].median(axis=1)

NameError: name 'concat_sub' is not defined

In [7]:
concat_sub.describe()

NameError: name 'concat_sub' is not defined

In [8]:
cutoff_lo = 0.8
cutoff_hi = 0.2

# Mean Stacking

In [9]:
concat_sub['scalar_coupling_constant'] = concat_sub['m_mean']
concat_sub[['id', 'scalar_coupling_constant']].to_csv('stack_mean.csv', 
                                        index=False, float_format='%.6f')

NameError: name 'concat_sub' is not defined

# Median Stacking

In [10]:
concat_sub['scalar_coupling_constant'] = concat_sub['m_median']
concat_sub[['id', 'scalar_coupling_constant']].to_csv('stack_median.csv', 
                                        index=False, float_format='%.6f')

NameError: name 'concat_sub' is not defined

# Pushout + Median Stacking
>* Pushout strategy is bit aggresive

In [11]:
concat_sub['scalar_coupling_constant'] = np.where(np.all(concat_sub.iloc[:,1:ncol] > cutoff_lo, axis=1), 1, 
                                    np.where(np.all(concat_sub.iloc[:,1:ncol] < cutoff_hi, axis=1),
                                             0, concat_sub['m_median']))
concat_sub[['id', 'scalar_coupling_constant']].to_csv('stack_pushout_median.csv', 
                                        index=False, float_format='%.6f')

NameError: name 'concat_sub' is not defined

# MinMax + Mean Stacking
>* MinMax seems more gentle and it outperforms the previous one

In [12]:
concat_sub['scalar_coupling_constant'] = np.where(np.all(concat_sub.iloc[:,1:ncol] > cutoff_lo, axis=1), 
                                    concat_sub['m_max'], 
                                    np.where(np.all(concat_sub.iloc[:,1:ncol] < cutoff_hi, axis=1),
                                             concat_sub['m_min'], 
                                             concat_sub['m_mean']))
concat_sub[['id', 'scalar_coupling_constant']].to_csv('stack_minmax_mean.csv', 
                                        index=False, float_format='%.6f')

NameError: name 'concat_sub' is not defined

# MinMax + Median Stacking

In [13]:
concat_sub['scalar_coupling_constant'] = np.where(np.all(concat_sub.iloc[:,1:ncol] > cutoff_lo, axis=1), 
                                    concat_sub['m_max'], 
                                    np.where(np.all(concat_sub.iloc[:,1:ncol] < cutoff_hi, axis=1),
                                             concat_sub['m_min'], 
                                             concat_sub['m_median']))
concat_sub[['id', 'scalar_coupling_constant']].to_csv('stack_minmax_median.csv', 
                                        index=False, float_format='%.6f')

NameError: name 'concat_sub' is not defined

## Averaging Ranks

In [14]:
concat_sub['scalar_coupling_constant'] = concat_sub['mol0'].rank(method ='min') + concat_sub['mol1'].rank(method ='min') + concat_sub['mol2'].rank(method ='min') 
concat_sub['scalar_coupling_constant'] = (concat_sub['scalar_coupling_constant']-concat_sub['scalar_coupling_constant'].min())/(concat_sub['scalar_coupling_constant'].max() - concat_sub['scalar_coupling_constant'].min())
concat_sub.describe()
concat_sub[['id', 'scalar_coupling_constant']].to_csv('stack_rank.csv', index=False, float_format='%.8f')

NameError: name 'concat_sub' is not defined

## Blend

In [15]:
sub1=pd.read_csv('../input/another-one/stackers_blend.csv')
sub3=pd.read_csv('../input/lgb-public-kernels-plus-more-features/sub_lgb_model_individual.csv')
temp=pd.read_csv('../input/another-one/stackers_blend.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../input/another-one/stackers_blend.csv'

In [16]:
sub2=pd.read_csv('../input/responsible-chemistry/submission.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../input/responsible-chemistry/submission.csv'

In [17]:
temp['scalar_coupling_constant'] = 0.6*sub3['scalar_coupling_constant'] + 0.4*sub1['scalar_coupling_constant']
temp.to_csv('submission4.csv', index=False )

NameError: name 'sub3' is not defined

In [18]:
sns.distplot(temp['scalar_coupling_constant'])

NameError: name 'temp' is not defined

submission4.csv gives best score.[ BLEND WINS ]
## <font color=blue>Vote early and vote often!</font>
